In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("drive/MyDrive/RAG_Hadith/my_scraped_data.csv")

In [4]:
import re

def normalize_arabic(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # tashkeel
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)

    # tatweel
    text = text.replace("ـ", "")

    # normalize alef
    text = re.sub(r"[أإآ]", "ا", text)

    # yaa
    text = text.replace("ى", "ي")

    # taa marbuta
    text = text.replace("ة", "ه")

    # punctuation
    text = re.sub(r"[^\w\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [5]:
df["hadith_norm"] = df["hadith"].apply(normalize_arabic)

In [6]:
df["hadith_norm"].shape

(226389,)

In [ ]:
df["hadith_norm"] = (
    df["hadith"]
    .fillna("")
    .str.strip()
)

print("Before:", len(df))

df_unique = df.drop_duplicates(subset=["hadith_norm"])

print("After:", len(df_unique))

Before: 226389
After: 152421


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2
)

X = vectorizer.fit_transform(df_unique["hadith_norm"])

In [ ]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(
    n_neighbors=10,
    metric="cosine"
)

nn.fit(X)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [ ]:
idx = 0

distances, indices = nn.kneighbors(X[idx])

print("Query Hadith:", df_unique.iloc[idx]["hadith"][:500])
for d, i in zip(distances[0], indices[0]):
    print("="*50)
    print("Similarity:", 1-d)
    print(df_unique.iloc[i]["hadith"][:500])

Query Hadith: - أتى رسولَ اللهِ صلَّى اللهُ عليهِ وسلَّمَ رجلٌ يَسوقُ بدَنتَه حافيًا فقالَ : اركَبْهَا . فركبَهَا
Similarity: 0.9999999999999998
- أتى رسولَ اللهِ صلَّى اللهُ عليهِ وسلَّمَ رجلٌ يَسوقُ بدَنتَه حافيًا فقالَ : اركَبْهَا . فركبَهَا
Similarity: 0.35275920036039543
- رأى النَّبيُّ صلَّى اللهُ عليه وسلَّم رجلًا يسوقُ بدَنةً قال: ( اركَبْها ) قال: إنَّها بدَنةٌ يا رسولَ اللهِ قال: ( اركَبْها ) قال: إنَّها بدَنةٌ يا رسولَ اللهِ قال: ( اركَبْها ) قال في الثَّالثةِ والرَّابعةِ: ( اركَبْها ويلَكَ )
Similarity: 0.3431999293998691
- مَرَّ رسولُ اللهِ صلَّى اللهُ عليه وسلَّمَ برَجُلٍ يَسوقُ بَدَنةً، فقال: اركَبْها. قال: يا رسولَ اللهِ، إنَّها بَدَنةٌ. قال: اركَبْها.
Similarity: 0.33829323248050835
- بينما رجلٌ يسوقُ بدَنةً مقلَّدةً فقال له رسولُ اللهِ صلَّى اللهُ عليه وسلَّم: ( اركَبْها ) قال: بَدَنةٌ يا رسولَ اللهِ! قال: ( اركَبْها ويلَكَ )
Similarity: 0.32308668565524523
- رأى رسولُ اللهِ صلَّى اللهُ عليه وسلَّم رجُلًا يَسوقُ بَدَنةً، قال: اركَبْها، قال: إنَّها بَدَنةٌ! قال: اركَبْ

## Semantic Clustring

In [1]:
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

In [2]:
model = SentenceTransformer(
    "Omartificial-Intelligence-Space/Arabic-Triplet-Matryoshka-V2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
df = pd.read_csv("../data/Hadith_Filtered_Books.csv")

In [ ]:
unique_hadiths = (
    df["hadith"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print(f"Unique hadiths: {len(unique_hadiths):,}")

embeddings = model.encode(
    unique_hadiths,
    batch_size=512,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)

emb_lookup = dict(zip(unique_hadiths, embeddings))

print("Embedding lookup created")

Unique hadiths: 113,083


Batches:   0%|          | 0/221 [00:00<?, ?it/s]

Embedding lookup created


In [22]:
import pickle

with open("drive/MyDrive/RAG_Hadith/emb_lookup.pkl", "wb") as f:
    pickle.dump(emb_lookup, f)

In [11]:
def deduplicate_hadiths(
    hadiths,
    emb_lookup,
    similarity_threshold=0.90,
):
    """
    Cluster similar hadiths and keep the longest hadith
    from each cluster.
    """

    hadiths = [ str(h).strip() for h in hadiths if pd.notna(h) and str(h).strip()]

    # exact dedup first
    hadiths = list(dict.fromkeys(hadiths))

    if len(hadiths) <= 1:
        return hadiths

    vectors = np.vstack(
        [emb_lookup[h] for h in hadiths]
    )

    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric="cosine",
        linkage="average",
        distance_threshold=1 - similarity_threshold,
    )

    labels = clustering.fit_predict(vectors)

    representatives = []

    for cluster_id in np.unique(labels):

        idxs = np.where(labels == cluster_id)[0]

        cluster_hadiths = [
            hadiths[i]
            for i in idxs
        ]

        # choose longest narration
        # TODO : make a list of priorities البخاري , مسلم , وهكذا و صحيح
        representative = max(
            cluster_hadiths,
            key=len
        )

        representatives.append(representative)

    return representatives

In [24]:
sharh_groups = (
    df.groupby("sharh")["hadith_norm"]
      .apply(list)
)

rows = []

for i, (sharh, hadiths) in enumerate(sharh_groups.items()):
  print(sharh)
  print("----------------------------------------")
  for j, hadith in enumerate(hadiths):
    print(f"{j}-  {hadith} \n\n\n")
  print("##############################################")
  if i == 1:
    break

"اللهم لك الحمد أنت كسوتنيه"، أي: أنت الذي رزقتني به من غير حول مني ولا قوة، "أسألك من خيره وخير ما صنع له"، أي: أعني على أن أستعمله في طاعتك وعبادتك، ويكون عونا لي فيهما، "وأعوذ بك من شره وشر ما صنع له"، أي: أن أعصي به أو يكون عونا لي في معصيتك. "قال أبو نضرة"، وهو المنذر بن مالك أحد التابعين: "فكان أصحاب النبي صلى الله عليه وسلم إذا لبس أحدهم ثوبا جديدا، قيل له"، أي: دعي له بقوله: "تبلي"، أي: تعمر فيه حتى يبلى الثوب ويهلك، "ويخلف الله تعالى"، أي: ويخلف الله عليك بعد إعمارك فيه بثوب آخر جديد.
----------------------------------------
0-  كان النبي صلي الله عليه وسلم اذا استجد ثوبا سماه قال اللهم انت كسوتني هذا القميص او الرداء او العمامه اسالك خيره وخير ما صنع له واعوذ بك من شره وشر ما صنع له 



1-  ان النبي صلي الله عليه وسلم كان اذا استجد ثوبا سماه باسمه فقال اللهم انت كسوتني هذا فلك الحمد اسالك من خيره وخير ما صنع له واعوذ بك من شره وشر ما صنع له 



2-  كان رسول الله صلي الله عليه وسلم اذا استجد ثوبا سماه باسمه عمامه او قميصا او رداء ثم يقول اللهم لك الحمد انت كسوتنيه اسالك خيره و

In [12]:
sharh_groups = (
    df.groupby("sharh")["hadith_norm"]
      .apply(list)
)

rows = []

for sharh, hadiths in sharh_groups.items():

    unique_hadiths = deduplicate_hadiths(
        hadiths,
        emb_lookup=emb_lookup,
        similarity_threshold=0.90,
    )

    rows.append(
        {
            "sharh": sharh,
            "original_hadith_count": len(hadiths),
            "unique_hadith_count": len(unique_hadiths),
            "hadiths": unique_hadiths,
        }
    )

result_df = pd.DataFrame(rows)

result_df.head()

,sharh,original_hadith_count,unique_hadith_count,hadiths
0,"""اللهم لك الحمد أنت كسوتنيه""، أي: أنت الذي رزق...",32,5,[كان رسول الله صلي الله عليه وسلم اذا استجد ثو...
1,"""حتى لو سألها نفسها"" للجماع ""وهي على ظهر قتب"" ...",2,2,[قدم معاذ اليمن او قال الشام فراي النصاري تسجد...
2,"""حنين"" واد بين مكة والطائف وقعت فيه الغزوة الت...",9,3,[كنا مع رسول الله صلي الله عليه وسلم في حنين ف...
3,"""وأيما امرئ ابتاع شاة فوجدها مصراة"" وصر البهيم...",5,3,[لا تباغضوا ولا تحاسدوا ولا تناجشوا ولا تدابرو...
4,"""وإن كان صلى إتماما لأربع""، إن صلى ما شك فيه ح...",4,2,[اذا شك احدكم في صلاته فليلق الشك وليبن علي ال...


In [13]:
print(
    "Original:",
    result_df["original_hadith_count"].sum()
)

print(
    "After clustering:",
    result_df["unique_hadith_count"].sum()
)

reduction = (
    1
    - result_df["unique_hadith_count"].sum()
    / result_df["original_hadith_count"].sum()
)

print(
    f"Reduction: {reduction:.1%}"
)

Original: 222599
After clustering: 64042
Reduction: 71.2%


In [23]:
idx = 0

row = result_df.iloc[idx]

print("=" * 80)
print("Sharh:")
print(row["sharh"][:1000])

print("\n")
print("=" * 80)
print("Representative Hadiths:")

for i, h in enumerate(row["hadiths"], start=1):
    print(f"\n[{i}]")
    print(h[:1000])

Sharh:
"اللهم لك الحمد أنت كسوتنيه"، أي: أنت الذي رزقتني به من غير حول مني ولا قوة، "أسألك من خيره وخير ما صنع له"، أي: أعني على أن أستعمله في طاعتك وعبادتك، ويكون عونا لي فيهما، "وأعوذ بك من شره وشر ما صنع له"، أي: أن أعصي به أو يكون عونا لي في معصيتك. "قال أبو نضرة"، وهو المنذر بن مالك أحد التابعين: "فكان أصحاب النبي صلى الله عليه وسلم إذا لبس أحدهم ثوبا جديدا، قيل له"، أي: دعي له بقوله: "تبلي"، أي: تعمر فيه حتى يبلى الثوب ويهلك، "ويخلف الله تعالى"، أي: ويخلف الله عليك بعد إعمارك فيه بثوب آخر جديد.


Representative Hadiths:

[1]
كان رسول الله صلي الله عليه وسلم اذا استجد ثوبا سماه باسمه اما قميصا او عمامه ثم يقول اللهم لك الحمد انت كسوتنيه اسالك من خيره وخير ما صنع له واعوذ بك من شره وشر ما صنع له قال ابو نضره وكان اصحاب النبي صلي الله عليه وسلم اذا لبس احدهم ثوبا جديدا قيل له تبلي ويخلف الله عز وجل

[2]
كان اصحاب النبي _صلي الله عليه واله وسلم_ اذا لبس احدهم ثوبا جديدا قيل له تبلي ويخلف الله تعالي

[3]
اللهم لك الحمد كما كسوتنيه اسالك خيره وخير ما صنع له واعوذ بك من شره وشر ما صنع ل

In [25]:
all_deduplicated_hadiths = []
for hadith_list in result_df['hadiths']:
    all_deduplicated_hadiths.extend(hadith_list)

df_deduplicated = df[df['hadith_norm'].isin(all_deduplicated_hadiths)].reset_index(drop=True)

print(f"Original DataFrame shape: {df.shape}")
print(f"Deduplicated DataFrame shape: {df_deduplicated.shape}")

display(df_deduplicated.head())

Original DataFrame shape: (226389, 12)
Deduplicated DataFrame shape: (117659, 12)


,page_id,url,categories,sharh,hadith,rawy,mohadth,source,page,hokm,takhrij,hadith_norm
0,161627,https://dorar.net/hadith/sharh/161627,آداب الطريق - المشي حافيا ، حج - الهدي ، آداب ...,أرسل الله تعالى نبيه محمدا صلى الله عليه وسلم ...,- أتى رسولَ اللهِ صلَّى اللهُ عليهِ وسلَّمَ رج...,أنس بن مالك,ابن حجر العسقلاني,المطالب العالية,2/49,هو في (الصحيح) من حديث أنس رضي الله عنه دون قو...,أخرجه أبو يعلى (2763),اتي رسول الله صلي الله عليه وسلم رجل يسوق بدنت...
1,161630,https://dorar.net/hadith/sharh/161630,جنة - صفة الجنة ، رقائق وزهد - التعرض لنفحات ر...,عيادة المريض حق من حقوق المسلم على أخيه المسلم...,- إذا عادَ المسلِمُ أخاه مشَى في خَرافةِ الجنّ...,علي بن أبي طالب,الألباني,صحيح الترغيب,3476,صحيح,أخرجه ابن ماجه (1442)، وأحمد (612) واللفظ له، ...,اذا عاد المسلم اخاه مشي في خرافه الجنه حتي يجل...
2,161631,https://dorar.net/hadith/sharh/161631,أدعية وأذكار - ما يقول من انتبه في نومه ، أدعي...,ذكر الله تعالى على كل حال وفي كل وقت في الصباح...,- إذا فزِع أحدُكُم في النَّومِ فلْيقُلْ : ( أع...,يحيى بن سعيد,الألباني,صحيح الترغيب,1601,حسن لغيره,أخرجه الترمذي (3528) واللفظ له، وابن أبي شيبة ...,اذا فزع احدكم في النوم فليقل اعوذ بكلمات الله ...
3,161634,https://dorar.net/hadith/sharh/161634,جنائز وموت - موت الأولاد وفضل احتسابهم ، جنة -...,إذا مات الولد صغيرا فإنه يكون سببا في دخول وال...,- أتحبُّهُ ؟ قال : نعَم يا رسولَ اللهِ ! أحبَّ...,قرة بن إياس المزني,الألباني,صحيح الترغيب,2007,صحيح,أخرجه أحمد (15633)، وابن حبان (2947)، والحاكم ...,اتحبه قال نعم يا رسول الله احبك الله كما احبه ...
4,161635,https://dorar.net/hadith/sharh/161635,جنة - صفة الجنة ، ملائكة - أعمال الملائكة ، إي...,حرص الإسلام على كل ما يقرب بين المسلمين، ومن ه...,- إذا عادَ الرَّجلُ أخاهُ المسلِمَ مشَى في خِر...,علي بن أبي طالب,الألباني,صحيح الجامع,682,صحيح,أخرجه أحمد (612)، وأبو يعلى (262),اذا عاد الرجل اخاه المسلم مشي في خرافه الجنه ح...


In [27]:
df_deduplicated['hadith_norm'].nunique()

62348